# Phân tích trạng thái kinh tế của các quốc gia
##### Author: Trần Hữu Sang
###### Note: tôi chọn dùng lạm phát, thất nghiệp, GDP làm biến đầu vào để đánh giá trạng thái kinh tế, không đủ để kết luận trực tiếp “khả năng phát triển kinh tế” của một quốc gia.
###### Source: https://www.macrotrends.net/


In [85]:
#%pip install -U undetected_chromedriver

In [ ]:
import pandas as pd
import requests
from io import StringIO
import undetected_chromedriver as uc
from pandas.api.types import is_string_dtype
from bs4 import BeautifulSoup
import time
UrlWeb = "https://www.macrotrends.net/global-metrics/countries/ranking"
Url_TyLeThatNghiep = "/unemployment-rate"
Url_TyLeLamPhat = "/inflation-rate-cpi"
Url_TangTruongGDP = "/gdp-growth-rate"
Url_GDPBinhQuan = "/gdp-per-capita"


In [87]:
def GetTableFromUrl(url, table_id="country_ranking"):
    driver = uc.Chrome(version_main=153)
    try:
        driver.get(UrlWeb + url)
        time.sleep(5)  # chờ Cloudflare verify + trang load xong

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        table = soup.find("table", id=table_id)

        if table is None:
            print("Không tìm thấy bảng — có thể bị chặn bởi Cloudflare || id sai.")
            with open("debug.html", "w", encoding="utf-8") as f:
                f.write(html)
            return None

        df = pd.read_html(str(table))[0]
        return df
    finally:
        driver.quit()

#### **Set Dataframe**

In [88]:
df_ThatNghiep = GetTableFromUrl(Url_TyLeThatNghiep)
df_ThatNghiep = df_ThatNghiep.melt(
    id_vars="Country Name",
    var_name="Year",
    value_name="Unemployment_Rate"
)
df_ThatNghiep

C:\Users\sang.th\AppData\Local\Temp\ipykernel_49820\2840444944.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


,Country Name,Year,Unemployment_Rate
0,Eswatini,2023,34.61%
1,South Africa,2023,32.10%
2,Djibouti,2023,26.07%
3,Botswana,2023,23.38%
4,Gabon,2023,20.16%
...,...,...,...
930,Cambodia,2019,0.12%
931,Qatar,2019,0.10%
932,West Bank And Gaza,2019,25.34%
933,Sudan,2019,11.70%


In [89]:
df_LeLamPhat = GetTableFromUrl(Url_TyLeLamPhat)
df_LeLamPhat = df_LeLamPhat.melt(
    id_vars="Country Name",
    var_name="Year",
    value_name="Inflation_Rate_Cpi"
)
df_LeLamPhat

Không tìm thấy bảng — có thể bị chặn bởi Cloudflare || id sai.


AttributeError: 'NoneType' object has no attribute 'melt'

In [ ]:
df_GDP_TT = GetTableFromUrl(Url_TangTruongGDP)
df_GDP_TT = df_GDP_TT.melt(
    id_vars="Country Name",
    var_name="Year",
    value_name="GDP_Growth_Rate"
)
df_GDP_TT

C:\Users\sang.th\AppData\Local\Temp\ipykernel_49820\2840444944.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


,Country Name,Year,GDP_Growth_Rate
0,Macao,2023,75.31%
1,Guyana,2023,33.80%
2,Samoa,2023,15.23%
3,Turks And Caicos Islands,2023,13.73%
4,Malta,2023,10.63%
...,...,...,...
1045,Estonia,2019,3.73%
1046,Marshall Islands,2019,10.45%
1047,Equatorial Guinea,2019,-5.48%
1048,Timor-Leste,2019,24.21%


In [ ]:
df_GDP_BQ = GetTableFromUrl(Url_GDPBinhQuan)
df_GDP_BQ = df_GDP_BQ.melt(
    id_vars="Country Name",
    var_name="Year",
    value_name="GDP_Per_Capita"
)
df_GDP_BQ

C:\Users\sang.th\AppData\Local\Temp\ipykernel_49820\2840444944.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


,Country Name,Year,GDP_Per_Capita
0,Monaco,2023,"$256,800"
1,Liechtenstein,2023,"$206,781"
2,Luxembourg,2023,"$133,231"
3,Bermuda,2023,"$132,962"
4,Ireland,2023,"$106,819"
...,...,...,...
1045,Guam,2019,"$39,275"
1046,St. Martin (French Part),2019,"$19,033"
1047,Northern Mariana Islands,2019,"$24,497"
1048,Syrian Arab Republic,2019,"$1,110"


## **Merge Dataframe**

In [ ]:
df = (
    df_ThatNghiep
    .merge(
        df_LeLamPhat,
        on=["Country Name", "Year"],
        how="inner"
    )
    .merge(
        df_GDP_TT,
        on=["Country Name", "Year"],
        how="inner"
    )
    .merge(
        df_GDP_BQ,
        on=["Country Name", "Year"],
        how="inner"
    )
)
df

,Country Name,Year,Unemployment_Rate,Inflation_Rate_Cpi,GDP_Growth_Rate,GDP_Per_Capita
0,Eswatini,2023,34.61%,0.00%,3.53%,"$3,756"
1,South Africa,2023,32.10%,6.08%,0.81%,"$6,034"
2,Djibouti,2023,26.07%,1.45%,6.81%,"$3,381"
3,Botswana,2023,23.38%,5.07%,3.21%,"$7,827"
4,Gabon,2023,20.16%,3.63%,2.44%,"$7,803"
...,...,...,...,...,...,...
845,Cambodia,2019,0.12%,1.94%,7.94%,"$2,226"
846,Qatar,2019,0.10%,-0.67%,0.69%,"$66,841"
847,West Bank And Gaza,2019,25.34%,1.58%,1.36%,"$3,657"
848,Sudan,2019,11.70%,50.99%,-2.18%,$710


#### **Cleaning data && change type**

In [ ]:

df

In [ ]:
def clean_data(col_name : str):
    if is_string_dtype(df[col_name]):
        df[col_name] = (
            df[col_name]
            .astype("string")
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
def ChanhgeType(col_name : str):
    df[col_name] = pd.to_numeric(
            df[col_name],
            errors="coerce"
        )
clean_data("GDP_Per_Capita")
ChanhgeType("Unemployment_Rate")
ChanhgeType("Inflation_Rate_Cpi")
ChanhgeType("GDP_Growth_Rate")
ChanhgeType("GDP_Per_Capita")

df

#### **Check missing data**

In [ ]:
def check_missing(df):
    missing_isnull = df.isnull().sum()
    missing_isna = df.isna().sum()
    result = pd.DataFrame({
        "Missing_IsNull": missing_isnull,
        "Missing_IsNan": missing_isna
    })
    return result

check_missing(df)

,Missing_IsNull,Missing_IsNan
Country Name,0,0
Year,0,0
Unemployment_Rate,0,0
Inflation_Rate_Cpi,0,0
GDP_Growth_Rate,0,0
GDP_Per_Capita,0,0


#### Create new column: GDP_PC_Score (Chỉ số mức sống), GDP_Growth_Score (Chỉ số tăng trưởng), Inflation_Score (Chỉ số ổn định giá), Unemployment_Score (Chỉ số thị trường lao động)

In [ ]:
df["GDP_PC_Score"] = (
    (df["GDP_Per_Capita"] - df["GDP_Per_Capita"].min())
    / (df["GDP_Per_Capita"].max() - df["GDP_Per_Capita"].min())
    * 100
)

,Country Name,Year,Unemployment_Rate,Inflation_Rate_Cpi,GDP_Growth_Rate,GDP_Per_Capita,GDP_PC_Score
0,Eswatini,2023,34.61%,0.00%,3.53%,3756,2.782923
1,South Africa,2023,32.10%,6.08%,0.81%,6034,4.470756
2,Djibouti,2023,26.07%,1.45%,6.81%,3381,2.505075
3,Botswana,2023,23.38%,5.07%,3.21%,7827,5.799238
4,Gabon,2023,20.16%,3.63%,2.44%,7803,5.781456
...,...,...,...,...,...,...,...
845,Cambodia,2019,0.12%,1.94%,7.94%,2226,1.649304
846,Qatar,2019,0.10%,-0.67%,0.69%,66841,49.524325
847,West Bank And Gaza,2019,25.34%,1.58%,1.36%,3657,2.709571
848,Sudan,2019,11.70%,50.99%,-2.18%,710,0.526058


In [ ]:
df["GDP_Growth_Score"] = (
    (df["GDP_Growth_Rate"] - df["GDP_Growth_Rate"].min())
    / (df["GDP_Growth_Rate"].max() - df["GDP_Growth_Rate"].min())
    * 100
)
df
